# 📥 Loading Dataset from a URL

When working with real-world datasets, they are often hosted on the internet (GitHub, Kaggle, APIs, etc.).  
Instead of downloading the file manually, we can **fetch it directly using Python** and load it into a Pandas DataFrame.

### 🔧 Libraries Used
| Library | Purpose |
|---------|---------|
| `requests` | Send HTTP GET request to download the file content |
| `io.StringIO` | Wrap raw text so Pandas can read it as if it were a file |
| `pandas` | Load the data into a structured DataFrame |

### 📌 Workflow
```
URL → requests.get() → response.text → StringIO() → pd.read_csv()
```

In [2]:
import  requests
from io import StringIO

import  pandas as pd

url = 'https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv'
headers = {'User-Agent': 'Mozilla/5.0'}
response = requests.get(url, headers=headers)

raw_data = response.text
data = StringIO(raw_data)

df = pd.read_csv(data)
print(df.head())

      crim    zn  indus  chas    nox     rm   age     dis  rad  tax  ptratio  \
0  0.00632  18.0   2.31     0  0.538  6.575  65.2  4.0900    1  296     15.3   
1  0.02731   0.0   7.07     0  0.469  6.421  78.9  4.9671    2  242     17.8   
2  0.02729   0.0   7.07     0  0.469  7.185  61.1  4.9671    2  242     17.8   
3  0.03237   0.0   2.18     0  0.458  6.998  45.8  6.0622    3  222     18.7   
4  0.06905   0.0   2.18     0  0.458  7.147  54.2  6.0622    3  222     18.7   

        b  lstat  medv  
0  396.90   4.98  24.0  
1  396.90   9.14  21.6  
2  392.83   4.03  34.7  
3  394.63   2.94  33.4  
4  396.90   5.33  36.2  


## 📄 Loading a TSV File from a URL

A **TSV (Tab-Separated Values)** file is similar to a CSV, but columns are separated by **tabs (`\t`)** instead of commas.

`pd.read_csv()` handles both formats — you just need to set the `sep` parameter:

| File Format | Separator | `sep` value |
|-------------|-----------|-------------|
| CSV | Comma `,` | `sep=','` *(default)* |
| TSV | Tab `\t` | `sep='\t'` |
| Pipe-delimited | Pipe `\|` | `sep='\|'` |

> **Note:** The code below uses a sample Kaggle TSV URL for illustration. Kaggle URLs require authentication in practice.

In [6]:
# read tsv file from url

import  requests
from io import StringIO

url='https://www.kaggle.com/datasets/kanak97/file.tsv'
headers = {'User-Agent': 'Mozilla/5.0'}

response = requests.get(url, headers=headers)
raw_data = response.text
data = StringIO(raw_data)
df = pd.read_csv(data, sep='\t')

print(df.head())

                                     <!DOCTYPE html>
0                                   <html lang="en">
1                                             <head>
2    <title>Kaggle: Your Home for Data Science</t...
3                           <meta charset="utf-8" />
4      <meta name="robots" content="index, follow...


# 📋 `pd.read_csv()` — Parameter Cheat Sheet

> A quick reference for the most useful parameters when loading CSV / Excel / TSV files into Pandas.  
> All examples below use `pd.read_csv()` but most parameters apply to `pd.read_excel()` too.

---

## 1. `names` — Assign Custom Column Names
Use when the file has **no header row**, or you want to **override** existing column names.

```python
# File content (no header):
# 1,23.5,Boston
# 2,18.0,Seattle

df = pd.read_csv('data.csv', header=None, names=['id', 'price', 'city'])
print(df.columns)  # Index(['id', 'price', 'city'], dtype='object')
```

---

## 2. `header` — Specify Which Row is the Header
By default Pandas treats **row 0** as the header. Change this if your header is on a different row.

```python
# header=0  → first row is column names (default)
# header=None → no header, use integer column indices
# header=2  → row index 2 is the header (rows 0,1 are skipped)

df = pd.read_csv('data.csv', header=2)
```

---

## 3. `usecols` — Load Only Specific Columns
Speeds up loading and saves memory when you only need a subset of columns.

```python
# By column name
df = pd.read_csv('data.csv', usecols=['age', 'salary', 'city'])

# By column index (0-based)
df = pd.read_csv('data.csv', usecols=[0, 2, 4])

# Using a lambda for flexible selection
df = pd.read_csv('data.csv', usecols=lambda col: col.startswith('sales'))
```

---

## 4. `skiprows` — Skip Rows at the Top
Use to skip metadata rows, blank lines, or comments at the beginning of a file.

```python
# Skip the first 3 rows
df = pd.read_csv('data.csv', skiprows=3)

# Skip specific row indices (0-based)
df = pd.read_csv('data.csv', skiprows=[0, 2, 5])

# Skip rows using a function (skip rows where first value is a comment '#')
df = pd.read_csv('data.csv', skiprows=lambda i: i > 0 and i % 2 == 0)
```

---

## 5. `nrows` — Read Only N Rows
Useful for **previewing** a large file without loading everything into memory.

```python
# Load only the first 100 rows
df = pd.read_csv('large_file.csv', nrows=100)
print(df.shape)  # (100, num_columns)
```

---

## 6. `encoding` — Handle Special Characters
Default encoding is `utf-8`. Use this when you see `UnicodeDecodeError`.

```python
# Common encodings
df = pd.read_csv('data.csv', encoding='utf-8')       # default
df = pd.read_csv('data.csv', encoding='latin-1')     # Western European
df = pd.read_csv('data.csv', encoding='iso-8859-1')  # same as latin-1
df = pd.read_csv('data.csv', encoding='cp1252')      # Windows default

# Tip: If unsure, try 'latin-1' — it rarely raises errors
```

---

## 7. `on_bad_lines` — Handle Corrupt / Malformed Lines
Some files have rows with too many/few fields. Control how Pandas handles them.

```python
# 'error'  → raise an error (default)
# 'warn'   → print a warning and skip the bad line
# 'skip'   → silently skip bad lines

df = pd.read_csv('messy_data.csv', on_bad_lines='skip')
df = pd.read_csv('messy_data.csv', on_bad_lines='warn')
df = pd.read_csv('messy_data.csv', on_bad_lines='error')
```

> ⚠️ In older Pandas (< 1.3): use `error_bad_lines=False` and `warn_bad_lines=True`

---

## 8. `low_memory` — Memory Optimization for Large Files
When `True` (default), Pandas processes the file in chunks to reduce memory use,  
but may guess column dtypes incorrectly causing a `DtypeWarning`.

```python
# Disable chunked processing for consistent dtype inference
df = pd.read_csv('large_file.csv', low_memory=False)

# Better practice: specify dtypes explicitly (see dtype section below)
```

---

## 9. `dtype` — Force Column Data Types
Prevents Pandas from guessing types and avoids unexpected type coercion.

```python
df = pd.read_csv('data.csv', dtype={
    'zip_code': str,       # keep leading zeros e.g. '07001'
    'age': int,
    'price': float,
    'category': 'category' # memory-efficient for repeated strings
})
```

---

## 10. `parse_dates` — Handling Dates
Automatically parse columns as datetime objects instead of strings.

```python
# Parse a single column
df = pd.read_csv('data.csv', parse_dates=['date'])

# Parse multiple columns into one datetime column
df = pd.read_csv('data.csv', parse_dates=[['year', 'month', 'day']])

# Combine with index
df = pd.read_csv('data.csv', parse_dates=True, index_col='date')

print(df['date'].dtype)  # datetime64[ns]
```

---

## 11. `na_values` — Handling Missing Values
Define what values should be treated as `NaN` (Not a Number / missing).

```python
# Treat these strings as NaN
df = pd.read_csv('data.csv', na_values=['NA', 'N/A', '--', 'missing', '?', ''])

# Per-column custom na values
df = pd.read_csv('data.csv', na_values={'age': [0, -1], 'city': ['unknown']})

# Check missing values
print(df.isnull().sum())
```

> **Default NaN values Pandas recognizes:** `''`, `'NA'`, `'NaN'`, `'null'`, `'None'`, `'N/A'`, `'na'`

---

## 12. `converters` — Apply Custom Functions to Columns
Transform column values during loading — useful for cleaning raw data on the fly.

```python
df = pd.read_csv('data.csv', converters={
    'price': lambda x: float(x.replace('$', '').replace(',', '')),
    'name':  lambda x: x.strip().title(),
    'flag':  lambda x: True if x == 'Y' else False
})
```

---

## 13. `chunksize` — Handling Large Files in Chunks
When a file is too large to fit in RAM, read it in chunks and process iteratively.

```python
chunk_list = []

for chunk in pd.read_csv('huge_file.csv', chunksize=10_000):
    # Process / filter each chunk
    filtered = chunk[chunk['value'] > 0]
    chunk_list.append(filtered)

df = pd.concat(chunk_list, ignore_index=True)
print(df.shape)
```

---

## 14. `sep` / `delimiter` — Different Delimiters

```python
pd.read_csv('file.csv')              # comma (default)
pd.read_csv('file.tsv', sep='\t')   # tab-separated
pd.read_csv('file.txt', sep='|')    # pipe-separated
pd.read_csv('file.txt', sep='\s+')  # any whitespace (regex)
```

---

## 15. `index_col` — Set a Column as Index

```python
# Use 'id' column as the row index
df = pd.read_csv('data.csv', index_col='id')

# Use the first column as index
df = pd.read_csv('data.csv', index_col=0)
```

---

## 🔑 Quick Reference Table

| Parameter | Type | What it does |
|-----------|------|--------------|
| `names` | list | Assign column names |
| `header` | int / None | Row to use as column names |
| `usecols` | list / callable | Select specific columns |
| `skiprows` | int / list | Skip rows from the top |
| `nrows` | int | Read only N rows |
| `encoding` | str | File character encoding |
| `on_bad_lines` | str | Handle malformed lines |
| `low_memory` | bool | Memory vs. dtype accuracy trade-off |
| `dtype` | dict | Force column data types |
| `parse_dates` | list / bool | Parse columns as datetime |
| `na_values` | list / dict | Custom missing value markers |
| `converters` | dict | Apply functions to columns on load |
| `chunksize` | int | Read file in chunks |
| `sep` | str | Column delimiter |
| `index_col` | str / int | Set row index column |